# 👔 Supervisor-Agent Pattern with LangGraph

## Orchestrating Specialized Agents with a Central Supervisor 🤖

This notebook demonstrates the **Supervisor pattern** — one of the most common multi-agent architectures. A central supervisor agent receives user requests and delegates them to specialized worker agents based on the task type.

### 🎯 What You'll Learn

1. **🏗️ Supervisor Architecture** - How a supervisor agent coordinates multiple specialized agents
2. **🔧 Tool-Equipped Agents** - Creating agents with specific tools (math operations, web search)
3. **🧠 Memory & Checkpointing** - Using `InMemorySaver` for conversation history within a thread
4. **📊 State Retrieval** - Inspecting checkpoints and state history after execution
5. **🔀 Output Modes** - Comparing `last_message` vs `full_history` output modes

### 🌟 Why the Supervisor Pattern?

The supervisor pattern is ideal when you have:
- **Distinct specializations** — each agent is an expert in one domain (math, research, coding, etc.)
- **Task routing needs** — the supervisor analyzes the request and picks the right expert
- **Conversation continuity** — the supervisor maintains context across multiple interactions

### 📋 Architecture Overview

```
         User Request
              │
        ┌─────▼─────┐
        │ Supervisor │  ← Routes tasks based on content
        └─────┬─────┘
         ┌────┴────┐
    ┌────▼───┐ ┌───▼────┐
    │  Math  │ │Research│
    │ Expert │ │ Expert │
    └────────┘ └────────┘
```

---

*💡 **Pro Tip**: The supervisor pattern scales well — you can add more specialized agents without changing the core architecture!*

![alt text](image-3.png)

## 1. 🔧 Environment Setup & Installation

### 📦 Required Packages

- **`langgraph-supervisor`** - High-level helper that creates a supervisor workflow with minimal code
- **`langchain_core`** - Foundation library providing message types and schema definitions
- **`langgraph`** - Core framework for stateful agent workflows
- **`langchain_openai`** - Integration with OpenAI's GPT models

### 🔑 Key Dependency: `langgraph-supervisor`
This package provides `create_supervisor()` — a convenience function that automatically builds a supervisor graph with tool-calling, message routing, and agent coordination built in.

In [ ]:
%%capture --no-stderr
# 📦 Install the langgraph-supervisor package for the supervisor pattern
%pip install langgraph-supervisor
# 📦 Install/upgrade core LangChain + LangGraph + OpenAI packages
%pip install --quiet -U langchain_core langgraph langchain_openai

## 2. 🔐 API Key Configuration

### Setting Up Your OpenAI Connection

The supervisor and worker agents all use OpenAI's GPT-4o model for reasoning and tool calling. This cell securely prompts for your API key if it's not already set in the environment.

In [ ]:
import os, getpass

def _set_env(var: str):
    """Helper to prompt for an environment variable if not already set."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

# 🔑 Prompt for the OpenAI API key (only if not already in environment)
_set_env("OPENAI_API_KEY")
print("✅ API key configured!")

## 3. 🤖 Create Specialized Agents & Supervisor

### 🏗️ Building the Multi-Agent System

In this section, we'll create the complete supervisor-agent system in one cell:

1. **📦 Import & configure** — Load all required classes and set up memory/model
2. **🔧 Define tools** — Create tool functions for math operations and web search
3. **🤖 Create worker agents** — Build specialized agents with their tools
4. **👔 Create supervisor** — Wire everything together with `create_supervisor()`
5. **⚡ Compile** — Produce the final runnable application

### 🔑 Key Components:

| Component | Purpose |
|---|---|
| **`InMemorySaver`** | Short-term memory — persists conversation within a thread |
| **`InMemoryStore`** | Long-term memory — persists data across multiple threads |
| **`create_agent()`** | Creates a ReAct-style agent with tools and a system prompt |
| **`create_supervisor()`** | Builds the supervisor graph that routes between agents |

### 💡 How the Supervisor Decides
The supervisor receives the user's message and uses **LLM reasoning** to decide which agent should handle it. It can also chain agents — e.g., research first, then do math on the results.

In [ ]:
# ============================================================
# 📦 Import all required classes
# ============================================================
from langchain_openai import ChatOpenAI

from langgraph_supervisor import create_supervisor  # High-level supervisor builder
from langchain.agents import create_agent           # Creates ReAct-style agents
from langgraph.checkpoint.memory import InMemorySaver  # Short-term memory (within thread)
from langgraph.store.memory import InMemoryStore       # Long-term memory (across threads)

# ============================================================
# 🧠 Set up memory components
# ============================================================
# InMemorySaver: Checkpointer that saves state at each step
#   - Enables conversation continuity within a single thread_id
#   - Lost when the process restarts (use SqliteSaver for persistence)
checkpointer = InMemorySaver()

# InMemoryStore: Key-value store for cross-thread data
#   - Lets agents store/retrieve information across different conversations
store = InMemoryStore()

# ============================================================
# 🤖 Initialize the LLM model
# ============================================================
# GPT-4o is used for both the supervisor and worker agents
# It handles reasoning, tool calling, and response generation
model = ChatOpenAI(model="gpt-4o")

# ============================================================
# 🔧 Define tools for the Math Expert agent
# ============================================================
# Each tool is a simple Python function with a docstring.
# LangChain automatically converts these into tool schemas
# that the LLM can call via function calling.

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

# ============================================================
# 🔧 Define tools for the Research Expert agent
# ============================================================
# This simulates a web search — in production, you'd connect
# to a real search API (e.g., Tavily, SerpAPI, Bing)

def web_search(query: str) -> str:
    """Search the web for information."""
    return (
        "Here are the headcounts for each of the FAANG companies in 2024:\n"
        "1. **Facebook (Meta)**: 67,317 employees.\n"
        "2. **Apple**: 164,000 employees.\n"
        "3. **Amazon**: 1,551,000 employees.\n"
        "4. **Netflix**: 14,000 employees.\n"
        "5. **Google (Alphabet)**: 181,269 employees."
    )

# ============================================================
# 🤖 Create the specialized worker agents
# ============================================================
# Each agent gets: a model, a list of tools, a name, and a system prompt
# The system prompt constrains the agent's behavior and role

# Math Expert: can add and multiply numbers
math_agent = create_agent(
    model=model,
    tools=[add, multiply],
    name="math_expert",
    system_prompt="You are a math expert. Always use one tool at a time."
)

# Research Expert: can search the web for information
research_agent = create_agent(
    model=model,
    tools=[web_search],
    name="research_expert",
    system_prompt="You are a world class researcher with access to web search. Do not do any math."
)

# ============================================================
# 👔 Create the Supervisor workflow
# ============================================================
# create_supervisor() builds a graph where:
#   - The supervisor LLM receives messages and decides which agent to call
#   - It can call agents multiple times and chain their outputs
#   - The prompt tells the supervisor what each agent is good at
workflow = create_supervisor(
    [research_agent, math_agent],     # List of worker agents
    model=model,                       # LLM for the supervisor's reasoning
    prompt=(                           # Instructions for the supervisor
        "You are a team supervisor managing a research expert and a math expert. "
        "For current events, use research_agent. "
        "For math problems, use math_agent."
    ),
    checkpointer=checkpointer,        # Enable conversation memory
)

# ============================================================
# ⚡ Compile the workflow into a runnable application
# ============================================================
# thread_id in config enables conversation continuity
config = {"configurable": {"thread_id": "6789"}}

# Compile with both checkpointer (short-term) and store (long-term) memory
app = workflow.compile(checkpointer=checkpointer, store=store)

print("✅ Supervisor-agent system created and compiled!")

### 🎨 Visualize the Supervisor Graph

Let's see how the supervisor connects to the worker agents. The graph shows the supervisor at the center with edges to each specialized agent.

In [ ]:
# 🎨 Render the supervisor graph as a visual diagram
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

## 4. 🧪 Test the Supervisor System

### 🚀 First Query: Combined Research + Math Task

This query requires **both** agents to work together:
1. The **research agent** looks up FAANG company headcounts
2. The **math agent** adds them together

Watch how the supervisor automatically routes the task to the right agents and chains their outputs!

In [ ]:
# ============================================================
# 🚀 Send a query that requires BOTH research and math
# ============================================================
# The supervisor should: 
#   1. Route to research_expert to find headcounts
#   2. Route to math_expert to sum them up
#   3. Return the combined answer
# ============================================================
result = app.invoke({
        "messages": [
            {
            "role": "user",
            "content": "what's the combined headcount of the FAANG companies in 2024?"
            }
        ]
    },
    config=config  # Uses thread_id "6789" for conversation continuity
)

In [ ]:
# 📋 Print all messages in the conversation to see the full agent interaction
# This shows: user message → supervisor routing → agent responses → final answer
for m in result['messages']:
    m.pretty_print()

### 🧠 Follow-Up Query: Testing Conversation Memory

Now we send a follow-up question using the **same `thread_id`**. Because we're using `InMemorySaver` as the checkpointer, the supervisor remembers the previous conversation and can answer follow-up questions without re-running the research.

### 💡 Why This Matters
Without checkpointing, each `invoke()` call would be stateless — the agent would have no memory of previous interactions. The checkpointer enables **multi-turn conversations**.

In [ ]:
# ============================================================
# 🧠 Follow-up query — tests conversation memory
# ============================================================
# This uses the SAME config (thread_id "6789"), so the supervisor
# remembers the FAANG headcount data from the previous turn.
# It should answer directly without calling research_expert again.
# ============================================================
result = app.invoke({
        "messages": [
            {
            "role": "user",
            "content": "what's number of employees in Meta?"
            }
        ]
    },
    config=config  # Same thread_id → conversation continues
)

In [ ]:
# 📋 Print the follow-up response — notice the supervisor may answer
# directly from memory without invoking the research agent again
for m in result['messages']:
    m.pretty_print()

## Last message only
![alt text](image-2.png)

## 5. 📊 Checkpointing & State Retrieval

### 🔍 Inspecting the Agent's Memory

LangGraph's checkpointing system lets you inspect the **complete state** of the workflow at any point. This is invaluable for:

- **Debugging** — See exactly what the agent knew at each step
- **Auditing** — Review the full decision trail for compliance
- **Replay** — Resume a conversation from any checkpoint
- **Testing** — Verify that state transitions are correct

### 🔑 Key Methods:
| Method | What It Returns |
|---|---|
| `get_state(config)` | The latest state snapshot for this thread |
| `get_state_history(config)` | All historical state snapshots (oldest to newest) |

In [ ]:
# ============================================================
# 📊 Inspect checkpoints and state history
# ============================================================

# Retrieve the LATEST state for this thread — shows current messages,
# agent outputs, and any metadata stored during the workflow
latest_state = app.get_state(config)
print(f"📌 Latest state: {latest_state.values}")

print("\n" + "=" * 60)

# Retrieve the FULL state history — every checkpoint from the
# beginning of this thread, showing how state evolved over time
state_history = app.get_state_history(config)
for i, snapshot in enumerate(state_history):
    print(f"\n📸 Checkpoint {i}: {snapshot.values}")

## 6. 🔀 Output Modes: Full History vs. Last Message

### 📋 Understanding Output Modes

`create_supervisor()` supports two output modes that control what messages are returned after execution:

| Mode | Behavior | Use Case |
|---|---|---|
| **`last_message`** (default) | Returns only the final supervisor response | Clean, user-facing responses |
| **`full_history`** | Returns ALL messages including internal agent chatter | Debugging, auditing, transparency |

### 💡 When to Use Each:
- Use **`last_message`** in production — users only see the final polished answer
- Use **`full_history`** during development — see every tool call, agent response, and routing decision

![alt text](image-1.png)

In [ ]:
# ============================================================
# 🔀 Create a supervisor with full_history output mode
# ============================================================
# This creates a SEPARATE supervisor instance configured to
# return ALL messages (including internal agent exchanges)
# instead of just the final response.
# ============================================================
message_history_workflow = create_supervisor(
    [research_agent, math_agent],
    model=model,
    prompt=(
        "You are a team supervisor managing a research expert and a math expert. "
        "For current events, use research_agent. "
        "For math problems, use math_agent."
    ),
    # 🔑 Key difference: output_mode='full_history' shows ALL messages
    # including tool calls, agent reasoning, and supervisor routing
    output_mode='full_history'
    #output_mode='last_message' # Default — only returns the final answer
)

# Compile WITHOUT checkpointer — this is a one-off query (no memory needed)
message_history_app = message_history_workflow.compile()

# Run the same FAANG headcount query to compare output
message_history_result = message_history_app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what's the combined headcount of the FAANG companies in 2024?"
        }
    ]
})

print("✅ Full history query complete!")

In [ ]:
# 📋 Print the FULL message history — you'll see every internal message
# including supervisor routing decisions, tool calls, and agent responses
# Compare this to the previous output where only the final message appeared
for m in message_history_result['messages']:
    m.pretty_print()

## 📝 Summary & Key Takeaways

### What We Built
A **supervisor-agent system** where a central supervisor LLM routes user requests to specialized worker agents (math expert and research expert), with full conversation memory and checkpointing.

### 🔑 Core Concepts Reviewed

| Concept | What We Learned |
|---|---|
| **Supervisor Pattern** | A central agent uses LLM reasoning to route tasks to specialists |
| **`create_supervisor()`** | High-level API that builds the routing graph automatically |
| **`create_agent()`** | Creates ReAct-style agents with tools and system prompts |
| **Checkpointing** | `InMemorySaver` enables multi-turn conversations within a thread |
| **State Retrieval** | `get_state()` and `get_state_history()` let you inspect agent memory |
| **Output Modes** | `full_history` shows all messages; `last_message` shows only the final answer |

### 🚀 Next Steps
- **Add more agents** — coding expert, data analyst, writing assistant
- **Use persistent memory** — Replace `InMemorySaver` with `SqliteSaver` for production
- **Try the Swarm pattern** — See the **Swarm-Agent** notebook for a peer-to-peer alternative!

---

*💡 The supervisor pattern is best when you need **centralized control** over task routing. For more autonomous, peer-to-peer coordination, check out the swarm pattern!*